# ROC, Lift, and Profit Curves

This notebook demonstrates the new curve plotting helpers in `cuanalytics`:
- `ca.plot_roc(...)`
- `ca.plot_lift(...)`
- `ca.plot_profit(...)`

We use one built-in dataset (`load_breast_cancer_data`) and compare multiple classifier models on the same test set.

In [ ]:
import cuanalytics as ca
import pandas as pd


## 1) Load data and split

In [ ]:
df = ca.load_breast_cancer_data()
train_df, test_df = ca.split_data(df, test_size=0.2, random_state=42)
df.head()

## 2) Fit a few classifiers

All models below expose `predict_proba(...)`, which the curve helpers use internally.

In [ ]:
logit = ca.fit_logit(train_df, formula='diagnosis ~ .', C=1.0, max_iter=2000)
tree = ca.fit_tree(train_df, formula='diagnosis ~ .', max_depth=4, criterion='entropy', random_state=42)
knn = ca.fit_knn_classifier(train_df, formula='diagnosis ~ .', k=9)

models = [logit, tree, knn]
model_names = ['Logistic', 'Tree (depth=4)', 'KNN (k=9)']

## 3) ROC Curve

In [ ]:
roc_out = ca.plot_roc(
    models=models,
    test_df=test_df,
    positive_class='M',
    model_names=model_names,
    title='Breast Cancer: ROC Comparison'
)


# Returned data is available in roc_out['data']
roc_out

## 4) Lift Curve

In [ ]:
lift_out = ca.plot_lift(
    models=models,
    test_df=test_df,
    positive_class='M',
    model_names=model_names,
    title='Breast Cancer: Lift Comparison'
)


# Example: top-decile lift summary
{name: round(info['top_decile_lift'], 3) for name, info in lift_out['data'].items()}


## 5) Profit Curve

`*_value` convention: use positive values for gains and negative values for losses/costs.

In [ ]:
profit_config = {
    'tp_value': 120.0,
    'fp_value': -15.0,
    'tn_value': 0.0,
    'fn_value': -180.0,
    'fixed_value': 0.0,
}

profit_out = ca.plot_profit(
    models=models,
    test_df=test_df,
    positive_class='M',
    model_names=model_names,
    profit_config=profit_config,
    title='Breast Cancer: Profit vs Threshold'
)


pd.DataFrame({
    name: {
        'max_profit': info['max_profit'],
        'best_threshold': info['max_profit_threshold']
    }
    for name, info in profit_out['data'].items()
}).T


## 6) Cumulative Response Curve

Shows cumulative capture of positives as you contact more of the population ranked by model score.

In [ ]:
cum_out = ca.plot_cumulative_response(
    models=models,
    test_df=test_df,
    positive_class='M',
    model_names=model_names,
    title='Breast Cancer: Cumulative Response Comparison'
)

# Example: show the first few rows for one model
first_name = model_names[0]
cum_out['data'][first_name]['data'].head()


## 7) Optional: single-model usage\n
\n
Each helper also accepts a single model object directly.

In [ ]:
# Optional: single-model usage with cutoff labels on the ROC curve
_single = ca.plot_roc(
    logit,
    test_df=test_df,
    positive_class='M',
    title='Logistic ROC (with cutoff labels)',
    show_cutoffs=True,
    cutoff_step=0.1,
    cutoff_fontsize=8,
)
_single

In [ ]:
logit.score(test_df, threshold=0.29, positive_class='M')
logit.score(test_df, threshold=0.5, positive_class='M')